# 🚀 Notebook 4 — Deployment and Multi-Agent System

**Author:** Maulik Mathur | **HuggingFace:** [maulik78](https://huggingface.co/maulik78)

---

## Why This Notebook Exists

A model that only runs in a Colab notebook is a research artifact.
A model that runs as a live API is an engineering product.

This notebook transforms the fine-tuned Llama model into a
production-grade system:

```
Notebook 3 output:           This notebook builds:
──────────────────────       ──────────────────────────────────
Model weights on HF     →   Serverless GPU API (Modal)
Colab inference code    →   Reusable agent classes
Manual testing          →   Automated deal detection pipeline
No notifications        →   Real-time phone alerts (Pushover)
```

## System Architecture

```
SpecialistAgent (fine-tuned Llama via Modal)
    ↓ 30% weight
EnsembleAgent ──────────────────────────────→ Final price
    ↑ 70% weight
FrontierAgent (Groq Llama 3.3-70B)

ScannerAgent (RSS feeds) → deals → EnsembleAgent → MessagingAgent → phone
```

## Why Two Agents in the Ensemble?

- **SpecialistAgent**: Knows Amazon pricing from fine-tuning but limited by
  small model size (3B) and lite training (20k items)
- **FrontierAgent**: Broader world knowledge from Groq's 70B model but
  no domain-specific fine-tuning
- **Ensemble**: Combines domain expertise + general reasoning

## Step 1 — Setup (Run Locally, Not Colab)

Week 8 runs on your local laptop. The agents call APIs — no GPU needed.

In [ ]:
import os
import sys
import re
import random
import logging
import requests
from dotenv import load_dotenv

# Load API keys from .env file
load_dotenv(override=True)

# Add project root to path
sys.path.insert(0, '..')

# Set up logging so we can see agent reasoning
# INFO level shows what each agent is doing
logging.basicConfig(
    level=logging.INFO,
    format='%(levelname)s | %(name)s | %(message)s'
)

print('Environment check:')
print(f'  HF_TOKEN:      {"✅" if os.environ.get("HF_TOKEN") else "❌ missing"}')
print(f'  GROQ_API_KEY:  {"✅" if os.environ.get("GROQ_API_KEY") else "❌ missing"}')
print(f'  PUSHOVER_USER: {"✅" if os.environ.get("PUSHOVER_USER") else "⚠️ optional"}')
print(f'  PUSHOVER_TOKEN:{"✅" if os.environ.get("PUSHOVER_TOKEN") else "⚠️ optional"}')

---
## Part 1 — Deploy the Model to Modal

Modal is a serverless GPU platform. We define a class, decorate it with
`@app.cls(gpu='T4')`, and Modal:
- Builds a Docker container with our dependencies
- Allocates a T4 GPU when needed
- Scales to zero when idle (no cost)
- Scales up on demand

The key insight: `@modal.enter()` loads the model ONCE when the container
starts. Subsequent calls to `price()` reuse the loaded model.

In [ ]:
# Show the Modal service code
# (Actually deployed via: modal deploy deployment/modal_service.py)

modal_service_code = '''
import modal
import os
import re
import torch

app = modal.App("pricer-service")

# Docker image with all ML dependencies
image = (
    modal.Image.debian_slim(python_version="3.11")
    .pip_install([
        "transformers>=4.47.0",
        "peft>=0.14.0",
        "bitsandbytes>=0.45.0",
        "accelerate>=1.3.0",
        "torch>=2.5.0",
        "huggingface_hub",
    ])
)

BASE_MODEL      = "meta-llama/Llama-3.2-3B"
FINETUNED_MODEL = "maulik78/pricer-2026-06-10_06.40.40-lite"

@app.cls(
    image=image,
    gpu="T4",                                    # cloud T4 GPU
    secrets=[modal.Secret.from_name("huggingface")],  # HF token
    timeout=600,
    scaledown_window=120,                         # stay warm 2 mins
)
class Pricer:

    @modal.enter()  # runs ONCE when container starts
    def setup(self):
        # This method loads the model into GPU memory.
        # It runs once at container startup (~30 seconds).
        # All subsequent .price() calls reuse the loaded model.
        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
        from peft import PeftModel
        from huggingface_hub import login

        login(token=os.environ["HF_TOKEN"])

        quant_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4"
        )

        self.tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
        self.tokenizer.pad_token    = self.tokenizer.eos_token
        self.tokenizer.padding_side = "right"

        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=quant_config,
            device_map="auto"
        )
        self.model = PeftModel.from_pretrained(base, FINETUNED_MODEL)
        self.model.eval()

    @modal.method()  # callable remotely via pricer.price.remote(text)
    def price(self, description: str) -> float:
        prompt  = f"What does this cost to the nearest dollar?\\n\\n{description}\\n\\nPrice is $"
        inputs  = self.tokenizer(prompt, return_tensors="pt").to("cuda")

        with torch.no_grad():
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                output_ids = self.model.generate(
                    **inputs,
                    max_new_tokens=8,
                    do_sample=False,
                    pad_token_id=self.tokenizer.eos_token_id
                )

        prompt_len = inputs["input_ids"].shape[1]
        generated  = output_ids[0, prompt_len:]
        result     = self.tokenizer.decode(generated, skip_special_tokens=True)

        cleaned = result.replace("$","").replace(",","").strip()
        match   = re.search(r"\\d+\\.?\\d*", cleaned)
        return float(match.group()) if match else 0.0
'''

print('Modal service code (see deployment/modal_service.py):')
print(modal_service_code[:500] + '...')
print('\nDeploy with:')
print('  cd deployment')
print('  modal deploy modal_service.py')

In [ ]:
# Test the deployed Modal service
import modal

Pricer = modal.Cls.from_name('pricer-service', 'Pricer')
pricer = Pricer()

test_description = """
Title: Sony WH-1000XM4 Wireless Headphones
Category: Electronics
Brand: Sony
Description: Industry leading noise cancelling wireless headphones
Details: 30 hour battery, touch sensor controls, Alexa built in
"""

print('Testing Modal deployment...')
print('(First call ~30 seconds — cold start with model loading)')
price = pricer.price.remote(test_description)
print(f'\nSony WH-1000XM4 → Predicted: ${price:.2f}')
print(f'Actual retail:              ~$280')
print(f'Error:                       ${abs(price - 280):.2f}')

---
## Part 2 — The Agent Classes

Each agent has ONE job. Clean interfaces. No hidden complexity.

In [ ]:
# Import our agent classes
sys.path.insert(0, '.')

from agents.price_agent import SpecialistAgent, FrontierAgent, EnsembleAgent
from agents.deal_scanner import DealScanner, Deal
from agents.notifier import Notifier

print('✅ All agent classes imported')

In [ ]:
# ── SPECIALIST AGENT ─────────────────────────────────────────
# Uses our fine-tuned Llama 3.2-3B via Modal
# Domain-specific: trained on 20k Amazon products
# Weakness: limited by small model + lite training

specialist = SpecialistAgent()

test_items = [
    "Title: DeWalt 20V MAX Cordless Drill\nCategory: Tools\nBrand: DeWalt\nDescription: Compact cordless drill\nDetails: Variable speed, LED light, 2 batteries",
    "Title: Lego Technic Bugatti Chiron\nCategory: Toys\nBrand: Lego\nDescription: 3599 piece advanced building set\nDetails: Working steering, W16 engine",
    "Title: Apple AirPods Pro (2nd Gen)\nCategory: Electronics\nBrand: Apple\nDescription: Wireless earbuds with active noise cancellation\nDetails: H2 chip, MagSafe charging case",
]

expected = [129, 350, 249]

print('SpecialistAgent (fine-tuned Llama 3.2-3B via Modal):')
print('-'*55)
for item, exp in zip(test_items, expected):
    price = specialist.price(item)
    error = abs(price - exp)
    print(f'  {item[:45]}...')
    print(f'  Expected ~${exp} | Predicted ${price:.0f} | Error ${error:.0f}')
    print()

In [ ]:
# ── FRONTIER AGENT ────────────────────────────────────────────
# Uses Groq's Llama 3.3-70B (free tier)
# Broader world knowledge, no domain fine-tuning
# Strength: understands brand prestige, market context

frontier = FrontierAgent()

print('FrontierAgent (Groq Llama 3.3-70B, zero-shot):')
print('-'*55)
for item, exp in zip(test_items, expected):
    price = frontier.price(item)
    error = abs(price - exp)
    print(f'  {item[:45]}...')
    print(f'  Expected ~${exp} | Predicted ${price:.0f} | Error ${error:.0f}')
    print()

In [ ]:
# ── ENSEMBLE AGENT ────────────────────────────────────────────
# Combines both agents: FrontierAgent × 0.7 + SpecialistAgent × 0.3
#
# WHY 70/30 weights:
#   FrontierAgent uses 70B parameter model vs 3B specialist
#   70B has much stronger reasoning about brand and market positioning
#   But specialist contributes domain-specific Amazon patterns
#   Weights chosen empirically — optimize on validation set for production

ensemble = EnsembleAgent()

print('EnsembleAgent (FrontierAgent×0.7 + SpecialistAgent×0.3):')
print('-'*55)
for item, exp in zip(test_items, expected):
    price = ensemble.price(item)
    error = abs(price - exp)
    print(f'  {item[:45]}...')
    print(f'  Expected ~${exp} | Predicted ${price:.0f} | Error ${error:.0f}')
    print()

In [ ]:
# Compare all three agents side by side
print('\nAgent Comparison:')
print('='*70)
print(f'{"Item":<30} {"Actual":>8} {"Specialist":>12} {"Frontier":>10} {"Ensemble":>10}')
print('-'*70)

for item, exp in zip(test_items, expected):
    sp = specialist.price(item)
    fr = frontier.price(item)
    en = ensemble.price(item)
    title = item.split('\n')[0].replace('Title: ','')[:28]
    print(f'{title:<30} ${exp:>7} ${sp:>11.0f} ${fr:>9.0f} ${en:>9.0f}')

print('='*70)
print('\nNote: Ensemble combines the strengths of both approaches')

---
## Part 3 — Deal Scanner

The DealScanner subscribes to RSS deal feeds, scrapes current deals,
then uses Groq's LLM to filter to the 5 best quality deals with
clear prices. This replaces manual deal hunting with automation.

In [ ]:
scanner = DealScanner()

print('Scanning for live deals...')
print('(Fetching from Slickdeals and DealNews RSS feeds)\n')

deals = scanner.scan()

print(f'Found {len(deals)} curated deals:')
print('='*60)
for i, deal in enumerate(deals, 1):
    print(f'{i}. {deal.title}')
    print(f'   Price: ${deal.price:.2f}')
    print(f'   URL: {deal.url[:60]}')
    print()

In [ ]:
# Show the LLM filtering logic
print('WHY we use LLM filtering (not just scraping):')
print('='*55)
print()
print('Raw RSS feed contains:')
print('  "$50 OFF Samsung TV" ← NOT a price, it is a discount amount')
print('  "Deal ends tonight" ← no product info')
print('  "Free shipping on orders over $25" ← not a product deal')
print('  "KitchenAid Mixer at Amazon for $349" ← this is a real deal!')
print()
print('LLM prompt tells the model to:')
print('  ✅ Only include deals with ACTUAL product price (not discount amount)')
print('  ✅ Rewrite description as product summary, not deal terms')
print('  ✅ Skip items where price is ambiguous')
print('  ✅ Return exactly 5 best quality deals in JSON format')
print()
print('Using structured output (JSON format) ensures parseable responses')

---
## Part 4 — Full Pipeline: Scan → Price → Alert

In [ ]:
notifier = Notifier()

# The deal threshold
# If deal price < estimated retail × (1 - THRESHOLD) → alert
# 0.4 means: alert if deal is 40%+ below estimated retail value
DISCOUNT_THRESHOLD = 0.40

print('🔍 Running full deal detection pipeline...')
print(f'   Alerting on deals with {int(DISCOUNT_THRESHOLD*100)}%+ discount\n')

alerts_sent = 0

for deal in deals:
    # Get estimated retail value from ensemble
    estimated_retail = ensemble.price(deal.description)
    
    # Calculate discount percentage
    if estimated_retail > 0:
        discount = (estimated_retail - deal.price) / estimated_retail
    else:
        discount = 0
    
    status = '🎉 ALERT' if discount >= DISCOUNT_THRESHOLD else '❌ skip'
    
    print(f'{status}: {deal.title[:50]}')
    print(f'  Deal price:      ${deal.price:.2f}')
    print(f'  Estimated retail: ${estimated_retail:.2f}')
    print(f'  Discount:         {discount*100:.0f}%')
    
    if discount >= DISCOUNT_THRESHOLD:
        notifier.notify(
            product=deal.title,
            deal_price=deal.price,
            estimated_value=estimated_retail,
            url=deal.url
        )
        alerts_sent += 1
    print()

print(f'Pipeline complete: {alerts_sent} alerts sent out of {len(deals)} deals')

---
## Part 5 — Project Summary

Let's see the complete picture of everything we built.

In [ ]:
print('='*65)
print('  COMPLETE PROJECT SUMMARY')
print('='*65)
print()
print('DATASET (HuggingFace):')
print('  maulik78/items_raw_full      → 820k curated Amazon products')
print('  maulik78/items_raw_lite      → 22k products (lite version)')
print('  maulik78/items_full          → 820k LLM-preprocessed products')
print('  maulik78/items_prompts_lite  → 22k training prompts')
print()
print('MODEL (HuggingFace):')
print('  maulik78/pricer-2026-06-10_06.40.40-lite')
print('  Base:     meta-llama/Llama-3.2-3B')
print('  Method:   QLoRA (4-bit NF4 + LoRA rank=32)')
print('  Training: 20k items, 1 epoch, T4 GPU, 1h 54min')
print('  MAE:      $58.74')
print()
print('DEPLOYMENT (Modal):')
print('  Serverless T4 GPU API')
print('  Endpoint: modal.com/apps/maulikalwar04/main/deployed/pricer-service')
print()
print('RESULTS:')
print(f'  {"Model":<35} {"MAE ($)":>8}')
print('  ' + '-'*45)
models = [
    ('Random Pricer (baseline floor)',   287.0),
    ('Constant Pricer (mean)',           141.0),
    ('Linear Regression (3 features)',   120.0),
    ('NL Linear Regression (BoW)',       100.0),
    ('Random Forest (15k subset)',         85.0),
    ('XGBoost (800k full dataset)',        68.23),
    ('Fine-tuned Llama 3.2-3B (ours)',    58.74),
]
for name, mae in models:
    marker = ' ← OUR MODEL' if 'ours' in name else ''
    bar    = '█' * int(mae/20)
    print(f'  {name:<35} ${mae:>7.2f}  {bar}{marker}')

print('='*65)
print()
print('KEY INSIGHT:')
print('  Traditional ML (XGBoost) sees: word counts')
print('  Fine-tuned LLM sees: semantic meaning')
print('  "Sony + wireless + noise-cancelling" → premium audio product')
print('  This semantic understanding gives 13.9% better accuracy')